# Retro compatibility summary

This notebook incrementally builds a CSV for completed runs in `output/retro_compatibility`. Runs already present in the CSV are skipped, so W&B history and local scores are processed only for newly completed runs. Metric maxima and their checkpoint values come from W&B history. Pareto-front and greedy hypervolume representative indices come from the local score archive at the checkpoint with the best `jack/mean_100`.

The notebook intentionally uses the project's `HVComputer`, backed by `moocore`, for hypervolume calculations. Run the final cell to create the CSV.

In [2]:
!nvidia-smi

Fri Sep 11 01:18:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.195.03             Driver Version: 570.195.03     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX 5000 Ada Gene...    On  |   00000000:55:00.0 Off |                  Off |
| 30%   37C    P8             14W /  250W |     194MiB /  32760MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import json
import math
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import wandb


def locate_retro_directory() -> tuple[Path, Path]:
    current = Path.cwd().resolve()
    for base in (current, *current.parents):
        candidates = [base] if base.name == "retro_compatibility" else []
        candidates.append(base / "output" / "retro_compatibility")
        for candidate in candidates:
            repo_root = candidate.parent.parent
            if (
                candidate.is_dir()
                and candidate.parent.name == "output"
                and (repo_root / "pyproject.toml").is_file()
            ):
                return repo_root, candidate
    raise FileNotFoundError(
        "Could not locate output/retro_compatibility from the current working directory."
    )


REPO_ROOT, RETRO_DIR = locate_retro_directory()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from genexp.mo.utils import HVComputer

WANDB_ENTITY = "jdg8833-new-york-university"
WANDB_PROJECT = "retro_compatibility"
NUM_REPRESENTATIVES = 4
CSV_PATH = RETRO_DIR / "retro_compatibility_summary.csv"

CSV_COLUMNS = [
    "run",
    "scalarization",
    "seed",
    "epoch",
    "ckpts",
    "objs",
    "jack_100_ckpt",
    "jack_100_mean",
    "jack_500_ckpt",
    "jack_500_mean",
    "full_hv_ckpt",
    "full_hv_val",
    "pareto_indices",
    "representative_indices",
    "representative_hv_val",
]
LEGACY_CSV_COLUMNS = [column for column in CSV_COLUMNS if column != "representative_hv_val"]

/home/juan.guevara/.conda/envs/diff/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Selection helpers

All objectives are maximized. Indices are mapped back to the original rows of the score archive (and therefore to the corresponding molecule-object ordering), not positions in the validity-filtered array.

In [4]:
def load_manifest(run_dir: Path) -> tuple[dict, dict]:
    manifest_path = run_dir / "run_manifest.json"
    try:
        with manifest_path.open() as manifest_file:
            manifest = json.load(manifest_file)
    except (OSError, json.JSONDecodeError) as exc:
        raise ValueError(f"Could not read manifest {manifest_path}: {exc}") from exc

    configuration = manifest.get("configuration")
    if not isinstance(configuration, dict):
        raise ValueError(f"Manifest {manifest_path} has no configuration object")
    return manifest, configuration


def completed_run_directories(retro_dir: Path) -> list[Path]:
    completed = []
    for run_dir in sorted(retro_dir.iterdir()):
        if not run_dir.is_dir() or not (run_dir / "run_manifest.json").is_file():
            continue
        manifest, _ = load_manifest(run_dir)
        if manifest.get("status") == "complete":
            completed.append(run_dir)
    if not completed:
        raise ValueError(f"No completed manifest-backed runs found in {retro_dir}")
    return completed


def best_history_value(wandb_run, metric: str) -> tuple[int, float]:
    candidates: list[tuple[int, float]] = []
    for history_row in wandb_run.scan_history(keys=["epoch", metric], page_size=1_000):
        epoch_raw = history_row.get("epoch")
        value_raw = history_row.get(metric)
        if epoch_raw is None or value_raw is None:
            continue
        try:
            epoch_value = float(epoch_raw)
            metric_value = float(value_raw)
        except (TypeError, ValueError):
            continue
        if not math.isfinite(epoch_value) or not epoch_value.is_integer():
            continue
        if math.isfinite(metric_value):
            candidates.append((int(epoch_value), metric_value))

    if not candidates:
        raise ValueError(f"Run {wandb_run.id} has no finite history values for {metric!r}")

    # Maximize the metric; for an exact tie, prefer the earliest checkpoint.
    return max(candidates, key=lambda item: (item[1], -item[0]))


def load_valid_scores(run_dir: Path, epoch: int) -> tuple[np.ndarray, np.ndarray]:
    score_path = run_dir / "scores" / f"score_{epoch}.npz"
    object_path = run_dir / "objs" / f"obj_{epoch}.pkl"
    if not score_path.is_file():
        raise FileNotFoundError(f"Missing score archive for best jack-100 checkpoint: {score_path}")
    if not object_path.is_file():
        raise FileNotFoundError(f"Missing corresponding molecule-object file: {object_path}")

    with np.load(score_path, allow_pickle=False) as archive:
        if "scores" not in archive or "validity" not in archive:
            raise ValueError(f"{score_path} must contain 'scores' and 'validity' arrays")
        scores = np.asarray(archive["scores"])
        validity = np.asarray(archive["validity"])

    if scores.ndim != 2 or scores.shape[1] != 2:
        raise ValueError(f"Expected {score_path} scores to have shape (N, 2), got {scores.shape}")
    if validity.ndim != 1 or validity.shape[0] != scores.shape[0]:
        raise ValueError(
            f"Expected {score_path} validity to have shape ({scores.shape[0]},), got {validity.shape}"
        )
    if validity.dtype != np.bool_:
        raise TypeError(f"Expected {score_path} validity to be boolean, got {validity.dtype}")

    original_indices = np.flatnonzero(validity)
    valid_scores = scores[validity]
    if not len(valid_scores):
        raise ValueError(f"{score_path} contains no valid scores")
    if not np.issubdtype(valid_scores.dtype, np.number) or not np.all(np.isfinite(valid_scores)):
        raise ValueError(f"Valid scores in {score_path} must be finite numeric values")
    return valid_scores, original_indices


def nondominated_mask_maximize(scores: np.ndarray) -> np.ndarray:
    if scores.ndim != 2:
        raise ValueError(f"scores must be two-dimensional, got shape {scores.shape}")
    mask = np.ones(len(scores), dtype=bool)
    for index, score in enumerate(scores):
        dominates = np.all(scores >= score, axis=1) & np.any(scores > score, axis=1)
        mask[index] = not np.any(dominates)
    return mask


def hypervolume_value(hv_computer: HVComputer, scores: torch.Tensor) -> float:
    value = float(hv_computer(scores).detach().cpu().item())
    if not math.isfinite(value):
        raise ValueError("HVComputer returned a non-finite hypervolume")
    return value


def greedy_hypervolume_indices(
    pareto_scores: np.ndarray,
    pareto_original_indices: np.ndarray,
    hv_computer: HVComputer,
    count: int,
) -> tuple[list[int], float]:
    if count <= 0:
        raise ValueError("count must be positive")
    if len(pareto_scores) < count:
        raise ValueError(
            f"Cannot select {count} representatives from a Pareto front of size {len(pareto_scores)}"
        )
    if len(pareto_scores) != len(pareto_original_indices):
        raise ValueError("Pareto scores and original indices must have equal length")

    rewards = torch.as_tensor(pareto_scores, dtype=torch.float32)
    selected: list[int] = []
    remaining = list(range(len(rewards)))
    selected_hypervolume = 0.0

    for _ in range(count):
        best_local_index = remaining[0]
        best_hypervolume = float("-inf")
        for candidate in remaining:
            candidate_subset = selected + [candidate]
            candidate_hypervolume = hypervolume_value(hv_computer, rewards[candidate_subset])
            if candidate_hypervolume > best_hypervolume:
                best_local_index = candidate
                best_hypervolume = candidate_hypervolume
        selected.append(best_local_index)
        remaining.remove(best_local_index)
        selected_hypervolume = best_hypervolume

    original_indices = [int(pareto_original_indices[index]) for index in selected]
    return original_indices, selected_hypervolume

## CSV builder

The `*_ckpt` fields below are checkpoint/epoch values from W&B, not checkpoint filenames. Runs without a local `ckpts` directory receive an empty `ckpts` field. `representative_hv_val` is the `moocore` hypervolume of the four greedily selected representatives.

In [5]:
def summarize_run(api: wandb.Api, run_dir: Path) -> dict[str, object]:
    manifest, configuration = load_manifest(run_dir)
    if manifest.get("status") != "complete":
        raise ValueError(f"Run is not complete: {run_dir}")

    print(f"Summarizing run {run_dir.name}...")
    wandb_run_id = manifest.get("wandb_run_id")
    if not isinstance(wandb_run_id, str) or not wandb_run_id:
        raise ValueError(f"Manifest for {run_dir.name} has no W&B run ID")
    if "seed" not in configuration:
        raise ValueError(f"Manifest for {run_dir.name} has no seed")

    wandb_path = f"{WANDB_ENTITY}/{WANDB_PROJECT}/{wandb_run_id}"
    try:
        wandb_run = api.run(wandb_path)
    except Exception as exc:
        raise RuntimeError(
            f"Could not fetch W&B run {wandb_path}. Confirm network access and run 'wandb login'."
        ) from exc

    jack_100_ckpt, jack_100_mean = best_history_value(wandb_run, "jack/mean_100")
    jack_500_ckpt, jack_500_mean = best_history_value(wandb_run, "jack/mean_500")
    full_hv_ckpt, full_hv_val = best_history_value(wandb_run, "full_hypervolume")

    valid_scores, valid_original_indices = load_valid_scores(run_dir, jack_100_ckpt)
    pareto_mask = nondominated_mask_maximize(valid_scores)
    pareto_scores = valid_scores[pareto_mask]
    pareto_original_indices = valid_original_indices[pareto_mask]

    invalid_value = float(configuration.get("invalid_val", -1.0))
    if not math.isfinite(invalid_value):
        raise ValueError(f"Manifest for {run_dir.name} has a non-finite invalid_val")
    hv_computer = HVComputer(
        ref_point=torch.full((2,), invalid_value, dtype=torch.float32),
        num_rew=2,
    )
    representative_indices, representative_hv_val = greedy_hypervolume_indices(
        pareto_scores,
        pareto_original_indices,
        hv_computer,
        NUM_REPRESENTATIVES,
    )

    if len(set(representative_indices)) != NUM_REPRESENTATIVES:
        raise AssertionError(f"Representative indices are not unique for {run_dir.name}")
    if not set(representative_indices).issubset(set(pareto_original_indices.tolist())):
        raise AssertionError(f"A representative is not on the Pareto front for {run_dir.name}")

    ckpts_dir = run_dir / "ckpts"
    objs_dir = run_dir / "objs"
    if not objs_dir.is_dir():
        raise FileNotFoundError(f"Missing object directory: {objs_dir}")

    return {
        "run": run_dir.name,
        "scalarization": configuration.get("scalarization") or "",
        "seed": int(configuration["seed"]),
        "epoch": jack_100_ckpt,
        "ckpts": str(ckpts_dir.resolve()) if ckpts_dir.is_dir() else "",
        "objs": str(objs_dir.resolve()),
        "jack_100_ckpt": jack_100_ckpt,
        "jack_100_mean": jack_100_mean,
        "jack_500_ckpt": jack_500_ckpt,
        "jack_500_mean": jack_500_mean,
        "full_hv_ckpt": full_hv_ckpt,
        "full_hv_val": full_hv_val,
        "pareto_indices": json.dumps([int(index) for index in pareto_original_indices]),
        "representative_indices": json.dumps(representative_indices),
        "representative_hv_val": representative_hv_val,
    }


def parse_representative_indices(value: object, run_name: str) -> list[int]:
    try:
        decoded = json.loads(str(value))
    except (TypeError, ValueError, json.JSONDecodeError) as exc:
        raise ValueError(f"Invalid representative_indices for {run_name}: {value!r}") from exc
    if (
        not isinstance(decoded, list)
        or len(decoded) != NUM_REPRESENTATIVES
        or any(isinstance(index, bool) or not isinstance(index, int) for index in decoded)
        or len(set(decoded)) != NUM_REPRESENTATIVES
    ):
        raise ValueError(
            f"Expected {NUM_REPRESENTATIVES} unique integer representative indices for {run_name}, "
            f"got {decoded!r}"
        )
    return decoded


def representative_hypervolume_from_existing_row(row: pd.Series) -> float:
    run_name = str(row["run"])
    run_dir = RETRO_DIR / run_name
    if not run_dir.is_dir() or run_dir.parent != RETRO_DIR:
        raise FileNotFoundError(f"Could not locate run directory for existing CSV row: {run_name}")

    epoch_raw = row["epoch"]
    try:
        epoch_value = float(epoch_raw)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"Invalid epoch for {run_name}: {epoch_raw!r}") from exc
    if not math.isfinite(epoch_value) or not epoch_value.is_integer():
        raise ValueError(f"Invalid epoch for {run_name}: {epoch_raw!r}")
    epoch = int(epoch_value)

    valid_scores, valid_original_indices = load_valid_scores(run_dir, epoch)
    valid_position_by_original = {
        int(original_index): valid_position
        for valid_position, original_index in enumerate(valid_original_indices)
    }
    representative_indices = parse_representative_indices(row["representative_indices"], run_name)
    try:
        representative_positions = [
            valid_position_by_original[index] for index in representative_indices
        ]
    except KeyError as exc:
        raise ValueError(
            f"Representative index {exc.args[0]} is not valid at epoch {epoch} for {run_name}"
        ) from exc

    _, configuration = load_manifest(run_dir)
    invalid_value = float(configuration.get("invalid_val", -1.0))
    if not math.isfinite(invalid_value):
        raise ValueError(f"Manifest for {run_name} has a non-finite invalid_val")
    hv_computer = HVComputer(
        ref_point=torch.full((2,), invalid_value, dtype=torch.float32),
        num_rew=2,
    )
    representative_scores = torch.as_tensor(
        valid_scores[representative_positions], dtype=torch.float32
    )
    return hypervolume_value(hv_computer, representative_scores)


def load_existing_summary(csv_path: Path) -> pd.DataFrame:
    if not csv_path.is_file():
        return pd.DataFrame(columns=CSV_COLUMNS)
    try:
        existing = pd.read_csv(csv_path)
    except (OSError, pd.errors.ParserError, pd.errors.EmptyDataError) as exc:
        raise ValueError(f"Could not read existing summary {csv_path}: {exc}") from exc
    existing_columns = list(existing.columns)
    if existing_columns == LEGACY_CSV_COLUMNS:
        existing["representative_hv_val"] = [
            representative_hypervolume_from_existing_row(row)
            for _, row in existing.iterrows()
        ]
        existing = existing[CSV_COLUMNS]
        temporary_path = csv_path.with_suffix(csv_path.suffix + ".tmp")
        existing.to_csv(temporary_path, index=False)
        temporary_path.replace(csv_path)
        print(f"Upgraded {csv_path} with representative_hv_val.")
    elif existing_columns != CSV_COLUMNS:
        raise ValueError(
            f"Existing CSV schema does not match this notebook. Expected {CSV_COLUMNS}, "
            f"got {existing_columns}"
        )
    duplicate_runs = existing.loc[existing["run"].duplicated(), "run"].tolist()
    if duplicate_runs:
        raise ValueError(f"Existing CSV contains duplicate runs: {duplicate_runs}")
    return existing


def build_summary_csv(csv_path: Path = CSV_PATH) -> pd.DataFrame:
    existing = load_existing_summary(csv_path)
    existing_runs = set(existing["run"].astype(str))
    new_run_dirs = [
        run_dir
        for run_dir in completed_run_directories(RETRO_DIR)
        if run_dir.name not in existing_runs
    ]
    if not new_run_dirs:
        print(f"No new completed runs; {csv_path} was left unchanged.")
        return existing

    try:
        api = wandb.Api()
    except Exception as exc:
        raise RuntimeError("Could not initialize the W&B API. Confirm network access and run 'wandb login'.") from exc

    rows = [summarize_run(api, run_dir) for run_dir in new_run_dirs]
    additions = pd.DataFrame(rows, columns=CSV_COLUMNS)
    write_header = not csv_path.exists() or csv_path.stat().st_size == 0
    additions.to_csv(csv_path, mode="a", header=write_header, index=False)
    print(f"Appended {len(additions)} new completed runs to {csv_path}.")
    return pd.concat([existing, additions], ignore_index=True)

## Generate the CSV

Running this cell reads any existing `retro_compatibility_summary.csv`, skips its recorded run names, and appends only newly completed runs. If the file has the previous schema without `representative_hv_val`, it receives a one-time local upgrade before append-only updates begin. If there are no new completed runs after that upgrade, W&B is not initialized.

In [6]:
summary_df = build_summary_csv()
print(f"Summary now contains {len(summary_df)} rows at {CSV_PATH}")
summary_df

No new completed runs; /home/juan.guevara/Code/flow-density-control/output/retro_compatibility/retro_compatibility_summary.csv was left unchanged.
Summary now contains 12 rows at /home/juan.guevara/Code/flow-density-control/output/retro_compatibility/retro_compatibility_summary.csv


,run,scalarization,seed,epoch,ckpts,objs,jack_100_ckpt,jack_100_mean,jack_500_ckpt,jack_500_mean,full_hv_ckpt,full_hv_val,pareto_indices,representative_indices,representative_hv_val
0,diffusion_blend_20260910_153136_855622,NaN,5,600,NaN,/home/juan.guevara/Code/flow-density-control/o...,600,3.320708,550,3.447635,600,3.499075,"[202, 285, 363, 364, 431, 434, 540, 675, 713, ...","[363, 540, 1036, 1662]",3.486928
1,diffusion_blend_20260910_170427_433467,NaN,5,500,NaN,/home/juan.guevara/Code/flow-density-control/o...,500,3.306045,350,3.400981,350,3.464684,"[29, 34, 66, 77, 80, 83, 187, 198, 200, 201, 2...","[898, 29, 365, 1146]",3.406422
2,diffusion_blend_20260910_171858_327955,NaN,5,350,NaN,/home/juan.guevara/Code/flow-density-control/o...,350,3.384096,350,3.477706,350,3.519136,"[2, 10, 19, 100, 120, 121, 123, 140, 168, 175,...","[988, 175, 1182, 274]",3.506251
3,geom_base_model_20260910_173819_303827,NaN,6,0,NaN,/home/juan.guevara/Code/flow-density-control/o...,0,3.140489,0,3.302239,0,3.373160,"[58, 64, 304, 319, 546, 703, 716, 778, 860, 10...","[1042, 58, 1070, 64]",3.361882
4,geom_base_model_20260910_173819_354273,NaN,5,0,NaN,/home/juan.guevara/Code/flow-density-control/o...,0,3.091703,0,3.232971,0,3.271812,"[22, 218, 233, 728, 736, 757, 834, 978, 1070, ...","[1088, 22, 1275, 1328]",3.262793
5,geom_base_model_20260910_174158_064049,NaN,7,0,NaN,/home/juan.guevara/Code/flow-density-control/o...,0,3.115637,0,3.271919,0,3.374780,"[39, 83, 99, 161, 173, 298, 359, 411, 462, 527...","[731, 359, 462, 83]",3.368153
6,hv_nft_20260904_231008_912634,sum,5,550,/home/juan.guevara/Code/flow-density-control/o...,/home/juan.guevara/Code/flow-density-control/o...,550,3.133487,400,3.295526,400,3.390001,"[4, 8, 10, 26, 35, 53, 64, 110, 115, 133, 154,...","[965, 4, 162, 26]",3.267004
7,hv_nft_20260904_231010_785991,improvement,5,450,/home/juan.guevara/Code/flow-density-control/o...,/home/juan.guevara/Code/flow-density-control/o...,450,3.401130,500,3.486749,250,3.525312,"[19, 37, 64, 74, 88, 94, 100, 127, 128, 160, 1...","[289, 160, 759, 427]",3.471232
8,hv_nft_20260905_054947_079203,sum,6,600,/home/juan.guevara/Code/flow-density-control/o...,/home/juan.guevara/Code/flow-density-control/o...,600,3.139044,600,3.297374,600,3.400007,"[2, 21, 32, 168, 207, 243, 251, 260, 270, 322,...","[207, 926, 251, 459]",3.393820
9,hv_nft_20260905_054949_928698,improvement,6,350,/home/juan.guevara/Code/flow-density-control/o...,/home/juan.guevara/Code/flow-density-control/o...,350,3.402066,350,3.500275,350,3.544628,"[19, 46, 87, 108, 114, 140, 144, 172, 173, 179...","[270, 172, 540, 46]",3.533798


In [9]:
def render_representatives(row) -> list[Path]:
    import pickle

    from experiments.render_molecule_3d import render_molecule_3d

    objs_dir = Path(row["objs"]).expanduser().resolve()
    run_dir = objs_dir.parent

    epoch_value = float(row["epoch"])
    if not math.isfinite(epoch_value) or not epoch_value.is_integer():
        raise ValueError(f"Invalid epoch for {row['run']}: {row['epoch']!r}")
    epoch = int(epoch_value)

    representatives = row["representative_indices"]
    if isinstance(representatives, str):
        representatives = json.loads(representatives)

    if (
        not isinstance(representatives, list)
        or any(
            isinstance(index, bool) or not isinstance(index, int)
            for index in representatives
        )
        or len(set(representatives)) != len(representatives)
    ):
        raise ValueError(
            f"Invalid representative_indices for {row['run']}: "
            f"{representatives!r}"
        )

    object_path = objs_dir / f"obj_{epoch}.pkl"
    if not object_path.is_file():
        raise FileNotFoundError(f"Missing molecule-object file: {object_path}")

    with object_path.open("rb") as molecule_file:
        molecules = pickle.load(molecule_file)

    if not isinstance(molecules, list):
        raise ValueError(f"Expected {object_path} to contain a molecule list")

    renders_dir = run_dir / "renders"
    renders_dir.mkdir(parents=True, exist_ok=True)

    rendered_paths = []
    for index in representatives:
        if index < 0 or index >= len(molecules):
            raise IndexError(
                f"Representative index {index} is outside the molecule range "
                f"[0, {len(molecules)}) for {row['run']}"
            )

        output_path = renders_dir / f"render_{index}.png"
        rendered_paths.append(
            render_molecule_3d(molecules[index], output_path)
        )

    return rendered_paths


rendered_by_run = {
    row["run"]: render_representatives(row)
    for _, row in summary_df.iterrows()
}

rendered_by_run

{'diffusion_blend_20260910_153136_855622': [PosixPath('/home/juan.guevara/Code/flow-density-control/output/retro_compatibility/diffusion_blend_20260910_153136_855622/renders/render_363.png'),
  PosixPath('/home/juan.guevara/Code/flow-density-control/output/retro_compatibility/diffusion_blend_20260910_153136_855622/renders/render_540.png'),
  PosixPath('/home/juan.guevara/Code/flow-density-control/output/retro_compatibility/diffusion_blend_20260910_153136_855622/renders/render_1036.png'),
  PosixPath('/home/juan.guevara/Code/flow-density-control/output/retro_compatibility/diffusion_blend_20260910_153136_855622/renders/render_1662.png')],
 'diffusion_blend_20260910_170427_433467': [PosixPath('/home/juan.guevara/Code/flow-density-control/output/retro_compatibility/diffusion_blend_20260910_170427_433467/renders/render_898.png'),
  PosixPath('/home/juan.guevara/Code/flow-density-control/output/retro_compatibility/diffusion_blend_20260910_170427_433467/renders/render_29.png'),
  PosixPath('/